# 1. Caricamento del dataset

In questo notebook vengono addestrati e confrontati diversi classificatori mediante la libreria Scikit-Learn.

L'obiettivo è individuare il modello che ottiene le migliori prestazioni sul test set e che verrà successivamente utilizzato sul file `real_settings.csv` fornito in sede d'esame.

In [7]:
import pandas as pd

training_df = pd.read_csv("../data/training.csv")

training_df.shape

(50000, 22)

# 2. Preparazione dei dati

Per il Task 5 vengono utilizzate tutte le feature disponibili nel dataset.

A differenza del Task 2, non è necessario effettuare discretizzazioni manuali poiché gli algoritmi di Scikit-Learn sono in grado di gestire direttamente variabili numeriche e categoriche codificate come interi.

In [8]:
X = training_df.drop(columns=["Diabetes_binary"])

y = training_df["Diabetes_binary"]

print(X.shape)
print(y.shape)

(50000, 21)
(50000,)


# 3. Suddivisione in Training Set e Test Set

Il dataset viene suddiviso in training set e test set.

Viene utilizzata una suddivisione stratificata per preservare la distribuzione delle classi presente nel dataset originale.

Questo approccio è particolarmente importante in presenza di dataset sbilanciati.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(40000, 21)
(10000, 21)


# 4. Baseline Model

Come riferimento iniziale viene considerato un classificatore banale che predice sempre la classe maggioritaria.

Questo permette di verificare se i modelli addestrati apportano un reale miglioramento.

In [10]:
y.value_counts(normalize=True) * 100

Diabetes_binary
0    86.066
1    13.934
Name: proportion, dtype: float64

## 5.0 Preparazione tabella risultati

Ogni classificatore aggiungerà una riga.

5. Addestramento dei classificatori

5.1 Naive Bayes

5.2 Decision Tree

5.3 Logistic Regression

5.4 k-NN

5.5 Perceptron


In [11]:
results_table = []

## 5.1 Naive Bayes

Il classificatore Naive Bayes assume indipendenza condizionata tra le feature dato il valore della classe.

Nonostante tale ipotesi sia spesso irrealistica, il modello risulta molto efficiente e rappresenta uno dei principali algoritmi probabilistici studiati durante il corso.

In [12]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

## Metriche

- Accuracy
- Precision
- Recall
- F1-Score

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [14]:
results_table.append({
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, nb_pred),
    "Precision": precision_score(y_test, nb_pred),
    "Recall": recall_score(y_test, nb_pred),
    "F1": f1_score(y_test, nb_pred)
})

In [15]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956


### Discussione dei risultati

Il classificatore Naive Bayes ottiene un'accuracy pari al 77.98%.

Tale valore appare inizialmente elevato, ma deve essere interpretato alla luce del forte sbilanciamento presente nel dataset. Infatti la classe negativa rappresenta circa l'86% delle osservazioni e un classificatore banale che predicesse sempre la classe maggioritaria raggiungerebbe un'accuracy superiore.

Per questo motivo risultano particolarmente importanti le metriche precision, recall e F1-score.

La recall pari al 57.93% indica che il modello riesce a identificare oltre la metà dei soggetti diabetici presenti nel test set.

La precision pari al 33.31% evidenzia invece la presenza di numerosi falsi positivi.

Nel complesso il modello rappresenta una buona baseline probabilistica, ma lascia ampio margine di miglioramento.

## 5.2 Decision Tree

Gli alberi decisionali sono modelli di classificazione che suddividono progressivamente il dataset in sottoinsiemi sempre più omogenei rispetto alla variabile target.

Ad ogni nodo viene selezionata la feature che massimizza la separazione tra le classi secondo una misura di impurità.

Uno dei principali vantaggi degli alberi decisionali è la loro elevata interpretabilità: il processo decisionale può essere rappresentato graficamente e tradotto in un insieme di regole IF-THEN.

In [16]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train, y_train) 

dt_pred = dt.predict(X_test)

In [17]:
results_table.append({
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, dt_pred),
    "Precision": precision_score(y_test, dt_pred),
    "Recall": recall_score(y_test, dt_pred),
    "F1": f1_score(y_test, dt_pred)
})

In [18]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972


### Confronto preliminare con Naive Bayes

Il Decision Tree ottiene un'accuracy leggermente superiore rispetto al Naive Bayes.


Analizzando recall e F1-score emerge che il Naive Bayes risulta attualmente più efficace nell'identificare i soggetti diabetici.

In particolare:

- il Decision Tree raggiunge una recall pari al 32.74%;
- il Naive Bayes raggiunge una recall pari al 57.93%.

Anche l'F1-score risulta superiore per il Naive Bayes.

Pertanto, nonostante la maggiore accuracy, il Decision Tree non può ancora essere considerato il modello migliore.

## 5.3 Logistic Regression

La regressione logistica è un modello lineare per la classificazione.

A differenza della regressione lineare, l'obiettivo non è predire un valore numerico continuo ma la probabilità di appartenenza ad una determinata classe.

Il modello utilizza la funzione logistica (sigmoide):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

che trasforma qualsiasi valore reale in una probabilità compresa tra 0 e 1.

Nel caso della classificazione binaria, la classe viene assegnata confrontando la probabilità stimata con una soglia decisionale (generalmente pari a $0.5$).

Poiché la regressione logistica è sensibile alle diverse scale delle feature, prima dell'addestramento viene effettuata una standardizzazione dei dati.

### Standardizzazione delle feature

Le feature del dataset presentano scale molto differenti.

Ad esempio:

- BMI varia circa tra 12 e 98;
- Age varia tra 1 e 13;
- Income varia tra 1 e 8;
- MentHlth e PhysHlth variano tra 0 e 30.

Per evitare che le variabili con valori numerici più elevati dominino il processo di apprendimento, viene applicata una standardizzazione.

La standardizzazione trasforma ogni feature in modo da avere media $\mu = 0$ e deviazione standard $\sigma = 1$, applicando a ciascun valore la trasformazione:

$$z = \frac{x - \mu}{\sigma}$$

dove $x$ è il valore originale, $\mu$ la media della feature e $\sigma$ la sua deviazione standard.

In [19]:
from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

In [21]:
results_table.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred),
    "Recall": recall_score(y_test, lr_pred),
    "F1": f1_score(y_test, lr_pred)
})

In [22]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706


### Analisi dei risultati

La Logistic Regression ottiene l'accuracy più elevata tra i modelli analizzati fino a questo momento (86.35%).


La recall risulta pari ad appena il 15.36%, valore significativamente inferiore rispetto a quello ottenuto dal Naive Bayes (57.93%).

Questo significa che il modello tende a classificare la maggior parte delle osservazioni come non diabetiche, riducendo il numero di falsi positivi ma aumentando notevolmente il numero di falsi negativi.

La precision (53.50%) è invece la più elevata tra i modelli analizzati: quando il modello predice la presenza di diabete, la previsione risulta spesso corretta.

Dal punto di vista pratico, il modello privilegia la precision rispetto alla recall. In un contesto sanitario ciò potrebbe non essere ideale, poiché perdere molti casi reali di diabete può essere più grave che produrre alcuni falsi allarmi.

Anche l'F1-score (23.87%) risulta inferiore a quello del Naive Bayes, indicando che il compromesso complessivo tra precision e recall non è ancora ottimale.

## 5.4 k-Nearest Neighbors (k-NN)

L'algoritmo k-NN appartiene alla categoria dei metodi Instance-Based Learning.

Per classificare una nuova osservazione, il modello identifica i k campioni più vicini nel training set e assegna la classe più frequente tra essi.

La vicinanza tra le osservazioni viene calcolata mediante una misura di distanza, generalmente la distanza euclidea.

Poiché il calcolo delle distanze è fortemente influenzato dalla scala delle feature, il modello viene addestrato utilizzando i dati standardizzati.

In [23]:
from sklearn.neighbors import KNeighborsClassifier

k = 5

knn = KNeighborsClassifier(
    n_neighbors=k
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

In [24]:
results_table.append({
    "Model": "k-NN (k=5)",
    "Accuracy": accuracy_score(y_test, knn_pred),
    "Precision": precision_score(y_test, knn_pred),
    "Recall": recall_score(y_test, knn_pred),
    "F1": f1_score(y_test, knn_pred)
})

In [25]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809


### Analisi dei risultati

Il classificatore k-NN ottiene un'accuracy pari all'85.19%, valore inferiore alla Logistic Regression ma superiore a Naive Bayes e Decision Tree.

La precision (43.45%) risulta relativamente elevata, indicando che una parte consistente delle osservazioni classificate come diabetiche appartiene effettivamente alla classe positiva.

Tuttavia la recall (20.96%) è piuttosto bassa: il modello riesce a individuare soltanto circa un quinto dei soggetti diabetici presenti nel test set.

Questo comportamento è tipico dei dataset sbilanciati. Poiché la maggior parte dei vicini appartiene alla classe non diabetica, il modello tende a favorire la classe maggioritaria.

L'F1-score (28.28%) risulta superiore a quello della Logistic Regression ma inferiore sia al Decision Tree sia al Naive Bayes.

Nel complesso il modello mostra buone capacità di classificazione generale ma una limitata sensibilità nell'identificazione dei casi di diabete.

## 5.5 Perceptron

Il Perceptron è uno dei più semplici algoritmi di apprendimento supervisionato per la classificazione binaria.

Il modello apprende iterativamente un iperpiano di separazione aggiornando i pesi delle feature quando viene commesso un errore di classificazione.

Dal punto di vista teorico rappresenta uno dei primi algoritmi di Machine Learning sviluppati per la classificazione lineare.

Poiché il processo di apprendimento dipende direttamente dai valori numerici delle feature, il modello viene addestrato utilizzando i dati standardizzati.

In [26]:
from sklearn.linear_model import Perceptron

perceptron = Perceptron(
    random_state=42,
    max_iter=1000
)

perceptron.fit(X_train_scaled, y_train)

perc_pred = perceptron.predict(X_test_scaled)

In [27]:
results_table.append({
    "Model": "Perceptron",
    "Accuracy": accuracy_score(y_test, perc_pred),
    "Precision": precision_score(y_test, perc_pred),
    "Recall": recall_score(y_test, perc_pred),
    "F1": f1_score(y_test, perc_pred)
})

In [28]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809
4,Perceptron,0.7898,0.297313,0.373295,0.330999


### Analisi dei risultati

Il Perceptron ottiene un'accuracy pari al 78.98%, un valore inferiore rispetto a Logistic Regression, k-NN e Decision Tree.

La precision (29.73%) risulta relativamente bassa, indicando che una parte significativa delle osservazioni classificate come diabetiche appartiene in realtà alla classe negativa.

La recall (37.33%) è invece superiore a quella ottenuta da Logistic Regression (15.36%), k-NN (20.96%) e Decision Tree (32.74%), ma inferiore rispetto al Naive Bayes (57.93%).

L'F1-score raggiunge il valore di 33.10%, collocandosi tra il Naive Bayes e gli altri modelli considerati.

Dal punto di vista teorico, il risultato è coerente con la natura del Perceptron: essendo un classificatore lineare, è in grado di apprendere soltanto frontiere decisionali lineari. Se il problema presenta relazioni più complesse tra le feature, il modello può risultare limitato rispetto ad approcci più flessibili.

Nel complesso il Perceptron mostra prestazioni discrete ma non rappresenta il modello migliore tra quelli analizzati.

# 6. Confronto dei risultati

In questa sezione vengono confrontate le prestazioni dei diversi classificatori addestrati sul dataset.

Poiché il dataset presenta un forte sbilanciamento delle classi, non è sufficiente considerare esclusivamente l'accuracy.

Per questo motivo vengono analizzate anche precision, recall e F1-score, che consentono di valutare più correttamente la capacità dei modelli di identificare i soggetti diabetici.

In [29]:
results_df = pd.DataFrame(results_table)

results_df

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809
4,Perceptron,0.7898,0.297313,0.373295,0.330999


## Discussione critica dei risultati

I risultati mostrano comportamenti differenti tra i classificatori analizzati.

La Logistic Regression ottiene la massima accuracy (86.35%) e la massima precision (53.50%), ma presenta una recall molto bassa (15.36%). Ciò significa che il modello tende a classificare la maggior parte delle osservazioni come non diabetiche, perdendo numerosi casi reali di diabete.

Il Naive Bayes ottiene invece la migliore recall (57.93%) e il miglior F1-score (42.30%). Questo indica una maggiore capacità di individuare i soggetti diabetici, mantenendo un equilibrio migliore tra precision e recall.

Decision Tree e Perceptron mostrano prestazioni intermedie, mentre il k-NN risulta penalizzato dallo sbilanciamento delle classi.

Considerando che il problema affrontato riguarda la diagnosi del diabete, la capacità di identificare correttamente i soggetti positivi assume particolare importanza. Per questo motivo il Naive Bayes rappresenta attualmente il modello più promettente tra quelli analizzati.

# 7. Cross Validation

La valutazione effettuata finora si basa su una singola suddivisione
Training/Test (Holdout Method).

Tuttavia le prestazioni ottenute potrebbero dipendere dalla particolare
suddivisione casuale dei dati.

Per ottenere una stima più robusta delle prestazioni dei classificatori,
viene applicata una Stratified 10-Fold Cross Validation.

In questo approccio:

- il dataset viene suddiviso in 10 blocchi (fold);
- ad ogni iterazione un fold viene utilizzato come validation set;
- i restanti 9 fold vengono utilizzati per l'addestramento;
- il processo viene ripetuto 10 volte;
- il risultato finale è ottenuto mediando le metriche sui 10 fold.

La versione stratificata garantisce che ogni fold mantenga
approssimativamente la stessa distribuzione delle classi presente
nel dataset originale.

In [30]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

# Creazione dello splitter
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


In [31]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

nb_scores = cross_val_score(
    GaussianNB(),
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("Media:", nb_scores.mean())
print("Deviazione standard:", nb_scores.std())

Media: 0.4145426485315652
Deviazione standard: 0.00964955245927612


In [32]:
# Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

lr_scores = cross_val_score(
    lr_pipeline,
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("Media:", lr_scores.mean())
print("Std:", lr_scores.std())

Media: 0.2327953841995683
Std: 0.014143215144132505


In [33]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt_scores = cross_val_score(
    DecisionTreeClassifier(random_state=42),
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("Media:", dt_scores.mean())
print("Std:", dt_scores.std())

Media: 0.30687148677662757
Std: 0.017711030613464088


In [34]:
# KNN
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

knn_scores = cross_val_score(
    knn_pipeline,
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("Media:", knn_scores.mean())
print("Std:", knn_scores.std())

Media: 0.2670939714862646
Std: 0.006583522291420424


In [35]:
# Perceptron
from sklearn.linear_model import Perceptron

perc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", Perceptron(max_iter=1000, random_state=42))
])

perc_scores = cross_val_score(
    perc_pipeline,
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("Media:", perc_scores.mean())
print("Std:", perc_scores.std())

Media: 0.3220878469653262
Std: 0.05439189784117197


In [36]:
cv_results = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Decision Tree",
        "Logistic Regression",
        "k-NN",
        "Perceptron"
    ],
    "CV_F1_Mean": [
        nb_scores.mean(),
        dt_scores.mean(),
        lr_scores.mean(),
        knn_scores.mean(),
        perc_scores.mean()
    ],
    "CV_F1_Std": [
        nb_scores.std(),
        dt_scores.std(),
        lr_scores.std(),
        knn_scores.std(),
        perc_scores.std()
    ]
})

cv_results.sort_values(
    by="CV_F1_Mean",
    ascending=False
)

,Model,CV_F1_Mean,CV_F1_Std
0,Naive Bayes,0.414543,0.009650
4,Perceptron,0.322088,0.054392
1,Decision Tree,0.306871,0.017711
3,k-NN,0.267094,0.006584
2,Logistic Regression,0.232795,0.014143


## Analisi dei risultati della Cross Validation

La Stratified 10-Fold Cross Validation conferma le conclusioni emerse
dalla precedente valutazione Holdout.

Il classificatore Naive Bayes ottiene il miglior F1-score medio
(0.4145), mantenendo inoltre una deviazione standard molto contenuta
(0.0096).

Questo risultato suggerisce che il modello è non soltanto accurato,
ma anche particolarmente stabile rispetto alle diverse suddivisioni
del dataset.

Il Perceptron raggiunge il secondo miglior F1-score medio
(0.3221), ma presenta la deviazione standard più elevata
(0.0544). Ciò indica una maggiore sensibilità ai dati utilizzati
per l'addestramento e quindi una minore stabilità.

Decision Tree e k-NN mostrano prestazioni intermedie.

La Logistic Regression conferma invece la difficoltà osservata
nelle precedenti valutazioni: nonostante l'elevata accuracy,
ottiene il peggior F1-score medio a causa della scarsa recall
sulla classe positiva.

Nel complesso la Cross Validation rafforza l'ipotesi che il
Naive Bayes rappresenti attualmente il modello più promettente
per il problema considerato.

# 8. Ottimizzazione degli iperparametri

I modelli addestrati nelle sezioni precedenti sono stati utilizzati
principalmente con i parametri predefiniti di Scikit-Learn.

Tuttavia le prestazioni di un classificatore possono dipendere in modo
significativo dalla scelta degli iperparametri.

In questa sezione vengono analizzati alcuni degli iperparametri più
importanti dei modelli che hanno mostrato maggiore potenziale:

- Decision Tree
- k-Nearest Neighbors
- Logistic Regression

L'obiettivo è individuare configurazioni in grado di migliorare
le prestazioni rispetto ai modelli iniziali.

## 8.1 Ottimizzazione del Decision Tree

Nel caso degli alberi decisionali, una profondità eccessiva può
portare a fenomeni di overfitting.

Per questo motivo vengono analizzati diversi valori del parametro
`max_depth`, che controlla la profondità massima dell'albero.

In [37]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

depths = [3, 5, 7, 10, 15, None]

dt_results = []

for d in depths:

    model = DecisionTreeClassifier(
        max_depth=d,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    dt_results.append({
        "max_depth": d,
        "F1_mean": scores.mean()
    })

pd.DataFrame(dt_results)

,max_depth,F1_mean
0,3.0,0.000000
1,5.0,0.212530
2,7.0,0.193983
3,10.0,0.239024
4,15.0,0.281124
5,NaN,0.306871


L'analisi del parametro `max_depth` mostra che la limitazione della
profondità dell'albero comporta una riduzione delle prestazioni.

Il miglior risultato viene ottenuto lasciando l'albero libero di
crescere (max_depth=None), con un F1-score medio pari a 0.307.

Ciò suggerisce che il problema richiede una struttura decisionale
relativamente complessa e che una potatura aggressiva provoca
underfitting.

## 8.2 Ottimizzazione del parametro k

Nel classificatore k-NN il parametro più importante è il numero
di vicini considerati.

Valori troppo piccoli possono produrre modelli instabili,
mentre valori troppo elevati possono portare ad una eccessiva
semplificazione.

In [38]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

k_values = [3, 5, 7, 9, 11, 15]

knn_results = []

for k in k_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    knn_results.append({
        "k": k,
        "F1_mean": scores.mean()
    })

pd.DataFrame(knn_results)

,k,F1_mean
0,3,0.290071
1,5,0.267094
2,7,0.245285
3,9,0.225374
4,11,0.212187
5,15,0.197350


Nel classificatore k-NN il valore ottimale risulta k=3.

All'aumentare del numero di vicini considerati si osserva una
progressiva riduzione dell'F1-score.

Questo comportamento è coerente con la presenza di una forte
classe maggioritaria nel dataset: utilizzando valori elevati di k,
la classe negativa tende a dominare il processo decisionale,
riducendo la capacità del modello di individuare i soggetti
diabetici.

## 8.3 Ottimizzazione del parametro C

Nella Logistic Regression il parametro C controlla il livello di
regolarizzazione.

Valori piccoli aumentano la regolarizzazione,
mentre valori elevati consentono al modello di adattarsi maggiormente
ai dati di addestramento.

In [39]:
from sklearn.linear_model import LogisticRegression

c_values = [0.01, 0.1, 1, 10, 100]

lr_results = []

for c in c_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier",
         LogisticRegression(
             C=c,
             max_iter=1000,
             random_state=42
         ))
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    lr_results.append({
        "C": c,
        "F1_mean": scores.mean()
    })

pd.DataFrame(lr_results)

,C,F1_mean
0,0.01,0.224996
1,0.10,0.231902
2,1.00,0.232795
3,10.00,0.232795
4,100.00,0.232795


La variazione del parametro C produce effetti molto limitati.

L'F1-score rimane sostanzialmente stabile per valori di C maggiori
o uguali a 1.

Questo suggerisce che le prestazioni della Logistic Regression sono
più influenzate dalla natura del dataset e dallo sbilanciamento
delle classi che dalla scelta del parametro di regolarizzazione.

# 9. Analisi Bias-Variance

Uno dei concetti fondamentali del Machine Learning è il compromesso tra
bias e variance.

- Un modello con elevato bias tende a semplificare eccessivamente il problema
  e può andare incontro a underfitting.

- Un modello con elevata variance tende invece ad adattarsi troppo ai dati
  di addestramento, rischiando fenomeni di overfitting.

Per valutare questo compromesso vengono utilizzati i risultati ottenuti
mediante Cross Validation.

In particolare:

- la media dell'F1-score fornisce una stima delle prestazioni attese del modello;
- la deviazione standard dell'F1-score misura la stabilità del modello
  rispetto ai diversi campioni di addestramento.

In [40]:
bias_variance_df = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Decision Tree",
        "Logistic Regression",
        "k-NN",
        "Perceptron"
    ],
    "Mean_F1": [
        0.4145,
        0.3069,
        0.2328,
        0.2671,
        0.3221
    ],
    "Std_F1": [
        0.0096,
        0.0177,
        0.0141,
        0.0066,
        0.0544
    ]
})

bias_variance_df

,Model,Mean_F1,Std_F1
0,Naive Bayes,0.4145,0.0096
1,Decision Tree,0.3069,0.0177
2,Logistic Regression,0.2328,0.0141
3,k-NN,0.2671,0.0066
4,Perceptron,0.3221,0.0544


## Interpretazione dei risultati

### Naive Bayes

Il classificatore Naive Bayes ottiene il miglior F1-score medio
(0.415) con una deviazione standard molto contenuta (0.010).

Questo indica che il modello è relativamente stabile e generalizza bene
sui diversi sottoinsiemi del dataset.

Il basso valore della deviazione standard suggerisce una variance ridotta.

---

### Decision Tree

Il Decision Tree ottiene un F1-score inferiore rispetto al Naive Bayes
(0.307) e una deviazione standard leggermente più elevata.

Il modello mostra una maggiore sensibilità alle variazioni del training set,
sebbene non emergano evidenze particolarmente forti di overfitting.

---

### Logistic Regression

La Logistic Regression presenta il valore medio di F1-score più basso
tra i modelli considerati (0.233).

La deviazione standard rimane contenuta, segno che il modello è stabile.

Tuttavia le prestazioni limitate suggeriscono la presenza di un elevato bias,
ossia una capacità insufficiente di rappresentare la complessità del problema.

---

### k-NN

Il classificatore k-NN presenta la deviazione standard più bassa
(0.007), risultando il modello più stabile.

Tuttavia l'F1-score medio rimane modesto (0.267).

Anche in questo caso emerge un comportamento tipico di modelli con bias elevato:
il modello è molto consistente ma non particolarmente efficace.

---

### Perceptron

Il Perceptron mostra la deviazione standard più elevata (0.054).

Questo indica una forte dipendenza dal campione di addestramento e quindi
una maggiore variance rispetto agli altri classificatori.

Le prestazioni risultano inoltre inferiori a quelle del Naive Bayes.


## Considerazioni conclusive

L'analisi Bias-Variance evidenzia che il classificatore Naive Bayes
rappresenta il miglior compromesso tra prestazioni e stabilità.

Pur basandosi sull'ipotesi semplificativa di indipendenza condizionata
delle feature, il modello ottiene:

- il miglior F1-score medio;
- una deviazione standard contenuta;
- una buona capacità di generalizzazione.

Al contrario:

- Logistic Regression e k-NN mostrano un bias elevato;
- il Perceptron evidenzia una variance maggiore;
- il Decision Tree ottiene prestazioni intermedie ma inferiori al Naive Bayes.

Pertanto, sulla base dei risultati ottenuti mediante Holdout,
Cross Validation e ottimizzazione degli iperparametri,
il classificatore Naive Bayes risulta il modello più equilibrato tra quelli analizzati.

# 10. Selezione del modello finale

Al termine della fase di addestramento e valutazione sono stati confrontati
cinque classificatori diversi:

- Naive Bayes
- Decision Tree
- Logistic Regression
- k-NN
- Perceptron

La valutazione è stata effettuata mediante:

- Holdout stratificato (training/test split);
- metriche Accuracy, Precision, Recall e F1-score;
- Cross Validation a 10 fold;
- ottimizzazione degli iperparametri;
- analisi Bias-Variance.

Poiché il dataset presenta un forte sbilanciamento delle classi
(circa l'86% di soggetti non diabetici e il 14% di soggetti diabetici),
l'accuracy non è stata considerata come unico criterio di scelta.

È stata invece privilegiata la metrica F1-score, che rappresenta
un compromesso tra precision e recall e risulta particolarmente
adatta ai problemi di classificazione sbilanciata.

In [41]:
final_results = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Decision Tree",
        "Logistic Regression",
        "k-NN",
        "Perceptron"
    ],
    "Mean_F1_CV": [
        0.4145,
        0.3069,
        0.2328,
        0.2671,
        0.3221
    ]
})

final_results.sort_values(
    by="Mean_F1_CV",
    ascending=False
)

,Model,Mean_F1_CV
0,Naive Bayes,0.4145
4,Perceptron,0.3221
1,Decision Tree,0.3069
3,k-NN,0.2671
2,Logistic Regression,0.2328


# 11. Conclusioni

L'obiettivo del progetto era costruire e valutare modelli di classificazione
per la previsione della presenza di diabete a partire da dati clinici e
comportamentali.

Il lavoro è stato articolato in diverse fasi.

Nella prima fase è stata effettuata un'attività di preprocessing,
comprendente l'analisi della qualità dei dati, la verifica dei range
delle feature e la costruzione dei file `manuale.csv` e `training.csv`.

Successivamente sono stati progettati e implementati manualmente due
classificatori studiati durante il corso:

- Naive Bayes;
- Decision Tree.

Questa fase ha consentito di comprendere nel dettaglio il funzionamento
interno degli algoritmi e il processo di costruzione delle regole
decisionali e delle probabilità condizionate.

Nella seconda parte del progetto sono stati addestrati diversi modelli
mediante la libreria Scikit-Learn:

- Naive Bayes;
- Decision Tree;
- Logistic Regression;
- k-NN;
- Perceptron.

Le prestazioni sono state valutate mediante training/test split,
Cross Validation e ottimizzazione degli iperparametri.

L'analisi dei risultati ha evidenziato che il dataset presenta un forte
sbilanciamento delle classi, rendendo insufficiente la sola accuracy
come criterio di valutazione.

Per questo motivo particolare attenzione è stata dedicata alle metriche
precision, recall e F1-score.

Tra tutti i modelli considerati, il classificatore Naive Bayes ha
ottenuto il miglior compromesso tra capacità predittiva e stabilità,
raggiungendo il valore più elevato di F1-score medio in Cross Validation.

Nonostante la sua ipotesi di indipendenza condizionata tra le feature,
spesso considerata semplificativa, il modello ha mostrato una buona
capacità di generalizzazione sul dataset analizzato.

Pertanto il classificatore Naive Bayes viene selezionato come modello
finale da utilizzare sul file `real_settings.csv` che verrà fornito
in sede d'esame.

Il progetto ha inoltre evidenziato l'importanza della valutazione
sperimentale dei modelli: algoritmi più complessi non garantiscono
necessariamente prestazioni migliori e la scelta finale deve sempre
essere supportata da un confronto quantitativo rigoroso.

## Sede d'Esame - 19/06/2026

## Caricamento Dati

In [42]:
real_df = pd.read_csv("../data/real settings.csv")

real_df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


## Verifica dati

In [43]:
real_df.shape

(304, 22)

In [61]:
real_df["Diabetes_binary"].value_counts()

Diabetes_binary
0.0    304
Name: count, dtype: int64

In [44]:
real_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Diabetes_binary       304 non-null    float64
 1   HighBP                304 non-null    float64
 2   HighChol              304 non-null    float64
 3   CholCheck             304 non-null    float64
 4   BMI                   304 non-null    float64
 5   Smoker                304 non-null    float64
 6   Stroke                304 non-null    float64
 7   HeartDiseaseorAttack  304 non-null    float64
 8   PhysActivity          304 non-null    float64
 9   Fruits                304 non-null    float64
 10  Veggies               304 non-null    float64
 11  HvyAlcoholConsump     304 non-null    float64
 12  AnyHealthcare         304 non-null    float64
 13  NoDocbcCost           304 non-null    float64
 14  GenHlth               304 non-null    float64
 15  MentHlth              3

In [45]:
real_df.isnull().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [46]:
real_df.duplicated().sum()

np.int64(0)

## Separazione delle feature

In [48]:
X_real = real_df.drop(columns="Diabetes_binary")

y_real = real_df["Diabetes_binary"]



print("X_real: ", X_real.shape)
print("y_real: ", y_real.shape)

X_real:  (304, 21)
y_real:  (304,)


## Applicazione di Naive Bayes

In [49]:
nb_pred_real = nb.predict(X_real)

In [62]:
result_table = []

In [63]:
result_table.append({
    "Model": "NB",
    "Accuracy": accuracy_score(y_real, nb_pred_real),
    "Precision": precision_score(y_real, nb_pred_real),
    "Recall": recall_score(y_real, nb_pred_real),
    "F1": f1_score(y_real, nb_pred_real)
})

/Users/yuvisingh1265/Desktop/Machine Learning/ML-Project-Uni2026/ML/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [64]:
pd.DataFrame(result_table)

,Model,Accuracy,Precision,Recall,F1
0,NB,0.746711,0.0,0.0,0.0


In [59]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_real,nb_pred_real)

array([[227,  77],
       [  0,   0]])